In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split
import time
import torch.nn.functional as F
import timm
from torchvision import models
from sklearn.metrics import accuracy_score, classification_report

from torchvision import transforms

IMAGE_SIZE = 224  

transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


test_ds      = datasets.ImageFolder("test", transform=transform)
test_loader  = DataLoader(test_ds, batch_size=32, shuffle=False)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


num_classes = len(test_ds.classes)
model_mvit = timm.create_model(
    "deit_base_distilled_patch16_224",
    pretrained=False,
    num_classes=num_classes
)

model_mvit.load_state_dict(torch.load("models/deit.pth", map_location=device))
model_mvit = model_mvit.to(device)
model_mvit.eval()


predictionresult = []

def soft_voting_ensemble(model1, loader):
    y_true, y_pred = [], []
    class_names = loader.dataset.classes  
    global_index = 0  

    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(device)
            labels = labels.to(device)
            out1 = model1(imgs)
            prob1 = F.softmax(out1, dim=1)
            preds = prob1.argmax(dim=1)
            y_true.extend(labels.cpu().numpy())
            y_pred.extend(preds.cpu().numpy())

    return y_true, y_pred


y_true, y_pred = soft_voting_ensemble(
    model_mvit,
    test_loader
)
acc = accuracy_score(y_true, y_pred)
print("Ensemble Accuracy:", acc)

print(classification_report(
    y_true,
    y_pred,
    target_names=test_ds.classes
))


/home/ai_atrlbcau/miniconda3/envs/tensor_torch/lib/python3.10/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: '/home/ai_atrlbcau/miniconda3/envs/tensor_torch/lib/python3.10/site-packages/torchvision/image.so: undefined symbol: _ZN3c1017RegisterOperatorsD1Ev'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(
/home/ai_atrlbcau/miniconda3/envs/tensor_torch/lib/python3.10/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(
/home/ai_atrlbcau/miniconda3/envs/tensor_torch/lib/python3.10/site-packages/torch/cuda/__init__.py:235: UserWarning: 
NVIDIA GeForce RTX 5060 with CUDA capability sm_120 is not compatible with the curr

Execution Time: 0.695984 seconds
Execution Time: 0.003781 seconds
Execution Time: 0.003798 seconds
Execution Time: 0.004469 seconds
Execution Time: 0.003936 seconds
Execution Time: 0.006795 seconds
Execution Time: 0.005178 seconds
Execution Time: 0.003818 seconds
Execution Time: 0.003876 seconds
Execution Time: 0.003813 seconds
Execution Time: 0.003788 seconds
Execution Time: 0.005625 seconds
Execution Time: 0.003886 seconds
Execution Time: 0.004154 seconds
Execution Time: 0.003781 seconds
Execution Time: 0.003951 seconds
Execution Time: 0.005619 seconds
Execution Time: 0.003914 seconds
Execution Time: 0.003844 seconds
Execution Time: 0.003665 seconds
Execution Time: 0.004242 seconds
Execution Time: 0.003708 seconds
Execution Time: 0.004367 seconds
Execution Time: 0.004388 seconds
Execution Time: 0.005700 seconds
Execution Time: 0.004007 seconds
Execution Time: 0.003746 seconds
Execution Time: 0.003744 seconds
Execution Time: 0.003883 seconds
Execution Time: 0.004364 seconds
Execution 